In [0]:
%sql
USE CATALOG nilay_healthcare_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%fs ls "abfss://raw@nilaystore.dfs.core.windows.net/labs"

path,name,size,modificationTime
abfss://raw@nilaystore.dfs.core.windows.net/labs/labs.csv,labs.csv,38904,1756525469000
abfss://raw@nilaystore.dfs.core.windows.net/labs/labs.json,labs.json,99666,1756525469000
abfss://raw@nilaystore.dfs.core.windows.net/labs/labs.parquet,labs.parquet,11809,1756525468000


I am able to connect to my ADLS account

In [0]:
%sql
-- Example for patients.csv
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.patients
USING CSV
OPTIONS (
  path 'abfss://raw@nilaystore.dfs.core.windows.net/patients/patients.csv',
  header 'true',
  inferSchema 'true'
);

In [0]:
%sql
-- Example for patients.csv
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.labs
USING CSV
OPTIONS (
  path 'abfss://raw@nilaystore.dfs.core.windows.net/labs/labs.csv',
  header 'true',
  inferSchema 'true'
);

In [0]:
%sql
-- Example for patients.csv
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.visits
USING CSV
OPTIONS (
  path 'abfss://raw@nilaystore.dfs.core.windows.net/visits/visits.csv',
  header 'true',
  inferSchema 'true'
);

In [0]:
%sql
-- Example for patients.csv
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.vitals
USING CSV
OPTIONS (
  path 'abfss://raw@nilaystore.dfs.core.windows.net/vitals/vitals.csv',
  header 'true',
  inferSchema 'true'
);


we don't have access to install the above library so its not possible to accept streaming data for this project. Instead of streaming data I am using batch vitals data for the anlaysis



In [0]:
# from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# vitals_schema = StructType([
#     StructField("patient_id", StringType(), True),
#     StructField("device_id", StringType(), True),
#     StructField("timestamp", StringType(), True),
#     StructField("vital_type", StringType(), True),
#     StructField("vital_value", DoubleType(), True)
# ])

# # Read Event Hub stream
# vitals_stream = (
#     spark.readStream
#     .format("eventhubs")
#     .options(**event_hub_conf)
#     .load()
# )

# # Parse JSON messages
# from pyspark.sql.functions import from_json, col

# json_df = (
#     vitals_stream
#     .select(from_json(col("body").cast("string"), vitals_schema).alias("data"))
#     .select("data.*")
# )
# display(json_df)




In [0]:
%sql
-- Example for patients.csv
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.patient_doctor_map
USING CSV
OPTIONS (
  path 'abfss://raw@nilaystore.dfs.core.windows.net/patient_doctor_map/patient_doctor_map.csv',
  header 'true',
  inferSchema 'true'
);

In [0]:
%sql
describe extended bronze.patients;

col_name,data_type,comment
patient_id,string,null
name,string,null
age,double,null
gender,string,null
diagnosis,string,null
prescription,string,null
,,
# Detailed Table Information,,
Catalog,nilay_healthcare_catalog,
Database,bronze,


In [0]:
%sql
show tables in bronze;

database,tableName,isTemporary
bronze,healthcare_log,false
bronze,labs,false
bronze,patient_doctor_map,false
bronze,patients,false
bronze,visits,false
bronze,vitals,false
,_sqldf,true


# Patient File cleaning  code

In [0]:
from pyspark.sql.functions import col, when

# Read the external table
patients_df = spark.table("nilay_healthcare_catalog.bronze.patients")
display(patients_df)

patient_id,name,age,gender,diagnosis,prescription
P00001,Patient_QU7Q,83.0,Other,Heart Disease,INVALID_28U
P00002,INVALID_1YN,75.0,F,Heart Disease,DrugA
P00003,Patient_T6WV,68.0,F,None,null
P00004,Patient_W3CB,48.0,Other,Diabetes,None
P00005,Patient_LVHI,55.0,INVALID_ONM,Diabetes,DrugA
P00006,Patient_QK99,null,Other,Asthma,DrugC
P00007,Patient_1SBU,31.0,F,Diabetes,DrugC
P00008,Patient_N84A,44.0,M,Hypertension,None
P00009,Patient_9OCW,52.0,F,Diabetes,DrugC
INVALID_EWG,Patient_L81E,49.0,INVALID_ONM,Diabetes,DrugA


In [0]:
print(patients_df.printSchema())

root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- prescription: string (nullable = true)

None


%md
# Patients Anomalies:
- `patient_id` contains null values and invalid IDs starting with **INVALID_**.  
- Some entries in the `name` column start with the prefix **INVALID_**, and some are null.  
- The `age` column contains outliers and null values. Data type should be converted to **INTEGER**.  
- Some values in the `gender` column start with **INVALID_**, and some are null.  
- The `diagnosis` column contains **INVALID_** values and some entries marked as **NONE**.  
- The `prescription` column contains null, **INVALID_**, and **NONE** values.  
  

In [0]:
from pyspark.sql.functions import col, when

# Step 1: Drop rows with missing patient_id
patients_clean = patients_df.dropna(subset=["patient_id"])

# Step 2: Clean patient_id
patients_clean = patients_clean.withColumn(
    "patient_id",
    when(col("patient_id").rlike("^INVALID_.*"), "Unknown_ID").otherwise(col("patient_id"))
)

# Step 3: Clean name
patients_clean = patients_clean.withColumn(
    "name",
    when(col("name").rlike("^INVALID_.*") | col("name").isNull(), "Unknown Name").otherwise(col("name"))
)

# Step 4: Clean age
patients_clean = (
    patients_clean
    .withColumn("age", col("age").cast("int"))
    .withColumn("age", when((col("age").isNull()) | (col("age") <= 0) | (col("age") > 120), 50).otherwise(col("age")))
)

# Step 5: Clean gender
patients_clean = patients_clean.withColumn(
    "gender",
    when(col("gender").isin("M", "F", "Other"), col("gender")).otherwise("Other")
)

# Step 6: Clean diagnosis
patients_clean = patients_clean.withColumn(
    "diagnosis",
    when(col("diagnosis").rlike("^INVALID_.*") | (col("diagnosis") == "NONE") | col("diagnosis").isNull(), "Undiagnosed")
    .otherwise(col("diagnosis"))
)

# Step 7: Clean prescription
patients_clean = patients_clean.withColumn(
    "prescription",
    when(col("prescription").rlike("^INVALID_.*") | (col("prescription") == "NONE") | col("prescription").isNull(), "None")
    .otherwise(col("prescription"))
)

# Step 8: Remove duplicates
patients_clean = patients_clean.dropDuplicates()

# Step 9: Standardize schema
patients_clean = patients_clean.select(
    col("patient_id").cast("string"),
    col("name").cast("string"),
    col("age").cast("int"),
    col("gender").cast("string"),
    col("diagnosis").cast("string"),
    col("prescription").cast("string")
)

# Write cleaned data as Delta table
patients_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nilay_healthcare_catalog.silver.patients_clean")

In [0]:
display(patients_clean)

patient_id,name,age,gender,diagnosis,prescription
P00024,Patient_R0L9,27,F,None,DrugC
P00134,Patient_GTCB,41,Other,Heart Disease,DrugC
P00020,Patient_U4WK,44,Other,Diabetes,DrugA
P00142,Patient_EQSO,24,F,Diabetes,DrugB
P00077,Patient_X0A1,23,Other,Heart Disease,None
P00177,Patient_KVRG,83,Other,Heart Disease,DrugA
P00162,Patient_S04B,32,M,None,DrugB
P00099,Patient_UKTM,53,Other,Diabetes,DrugC
P00045,Patient_3C2G,60,Other,Heart Disease,None
P00161,Patient_U5GF,20,Other,Hypertension,None


# Patient_doctor_map cleaning code

In [0]:
from pyspark.sql.functions import col, when

# Read raw doctor-patient mapping table
doctor_map_df = spark.table("nilay_healthcare_catalog.bronze.patient_doctor_map")
display(doctor_map_df)

patient_id,doctor_id,care_team
null,D005,Endocrine
P00002,D005,General
P00003,D003,Pulmonary
P00004,D004,Pulmonary
P00005,D004,Endocrine
P00006,null,Cardio
P00007,D003,Endocrine
P00008,D004,Endocrine
P00009,D003,General
P00010,D001,Cardio


In [0]:
print(doctor_map_df.printSchema())

root
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- care_team: string (nullable = true)

None


# Patient_doctor_map Anomalies
- `patient_id` contains null values and invalid IDs starting with **INVALID_**.
- `doctor_id` contains null values and invalid IDs starting with **INVALID_**.
- `care_team` contains null values and invalid IDs starting with **INVALID_**.

In [0]:
# Step 1: Drop rows with null or invalid patient_id
doctor_map_clean = doctor_map_df.filter(
    (col("patient_id").isNotNull()) & (~col("patient_id").rlike("^INVALID_.*"))
)

# Step 2: Drop rows with null or invalid doctor_id
doctor_map_clean = doctor_map_clean.filter(
    (col("doctor_id").isNotNull()) & (~col("doctor_id").rlike("^INVALID_.*"))
)

# Step 3: Clean care_team → replace null/invalid with "Unassigned"
doctor_map_clean = doctor_map_clean.withColumn(
    "care_team",
    when((col("care_team").isNull()) | (col("care_team").rlike("^INVALID_.*")), "Unassigned")
    .otherwise(col("care_team"))
)

# Step 4: Remove duplicates
doctor_map_clean = doctor_map_clean.dropDuplicates()

# Step 5: Enforce schema
doctor_map_clean = doctor_map_clean.select(
    col("patient_id").cast("string"),
    col("doctor_id").cast("string"),
    col("care_team").cast("string")
)

# Step 6: Save cleaned data into Silver (Delta)
doctor_map_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nilay_healthcare_catalog.silver.patient_doctor_map_clean")


In [0]:
%sql
show tables in silver

database,tableName,isTemporary
silver,labs_clean,false
silver,patient_doctor_map_clean,false
silver,patients_clean,false
silver,visits_clean,false
silver,vitals_clean,false
,_sqldf,true


In [0]:
display(doctor_map_clean)

patient_id,doctor_id,care_team
P00112,D004,Cardio
P00074,D002,Endocrine
P00143,D003,Cardio
P00190,D003,Cardio
P00051,D002,General
P00085,D001,General
P00133,D004,Endocrine
P00186,D004,Endocrine
P00091,D005,Endocrine
P00018,D004,Cardio


# Labs cleaning code

In [0]:
from pyspark.sql.functions import col, when

labs_df = spark.table("nilay_healthcare_catalog.bronze.labs")
display(labs_df)
print(labs_df.printSchema())


patient_id,test_name,result,unit,date
P00054,Lipid Panel,4.15,x10^9/L,2025-08-09
P00121,Lipid Panel,9.49,%,2025-07-05
P00034,Lipid Panel,2.56,%,2025-07-04
P00185,ECG,8.94,x10^9/L,2025-06-24
P00128,Lipid Panel,2.87,%,2025-08-26
P00135,COVID19 PCR,4.24,%,2025-07-22
P00066,ECG,1.63,INVALID_756,2025-06-21
P00198,COVID19 PCR,6.88,INVALID_756,2025-08-26
P00001,HbA1c,1.3,%,2025-06-17
P00107,HbA1c,8.23,mg/dL,2025-06-15


root
 |-- patient_id: string (nullable = true)
 |-- test_name: string (nullable = true)
 |-- result: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- date: string (nullable = true)

None


# Anomalies
- `patient_id` contains null values and invalid IDs starting with **INVALID_**.
- `test_name` contains null values and invalid IDs starting with **INVALID_**.
- `result` contains null values and invalid IDs starting with **INVALID_**.
- `unit` contains null values and invalid IDs starting with **INVALID_**.
- `date` contains null values and invalid IDs starting with **INVALID_**.

In [0]:
from pyspark.sql.functions import col, when, to_date

# Read raw labs data
labs_df = spark.table("nilay_healthcare_catalog.bronze.labs")

# Step 1: Drop rows with null or invalid patient_id
labs_clean = labs_df.filter(
    (col("patient_id").isNotNull()) & (~col("patient_id").rlike("^INVALID_.*"))
)

# Step 2: Clean test_name
labs_clean = labs_clean.withColumn(
    "test_name",
    when((col("test_name").isNull()) | (col("test_name").rlike("^INVALID_.*")), "Unknown_Test")
    .otherwise(col("test_name"))
)

# Step 3: Clean result
labs_clean = labs_clean.withColumn(
    "result",
    when((col("result").isNull()) | (col("result").rlike("^INVALID_.*")), "Unknown_Result")
    .otherwise(col("result"))
)

# Step 4: Clean unit
labs_clean = labs_clean.withColumn(
    "unit",
    when((col("unit").isNull()) | (col("unit").rlike("^INVALID_.*")), "Unknown_Unit")
    .otherwise(col("unit"))
)

# Step 5: Clean date (drop invalid/null, cast to proper date)
labs_clean = labs_clean.filter(
    (col("date").isNotNull()) & (~col("date").rlike("^INVALID_.*"))
)
labs_clean = labs_clean.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# Step 6: Remove duplicates
labs_clean = labs_clean.dropDuplicates()

# Step 7: Enforce schema
labs_clean = labs_clean.select(
    col("patient_id").cast("string"),
    col("test_name").cast("string"),
    col("result").cast("string"),
    col("unit").cast("string"),
    col("date").cast("date")
)

# Step 8: Save cleaned data into Silver (Delta)
labs_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nilay_healthcare_catalog.silver.labs_clean")

In [0]:
display(labs_clean)

patient_id,test_name,result,unit,date
P00158,HbA1c,Unknown_Result,Unknown_Unit,2025-06-03
P00128,Lipid Panel,2.87,%,2025-08-26
P00168,ECG,5.67,x10^9/L,2025-07-17
P00095,Unknown_Test,6.47,%,2025-06-16
P00065,HbA1c,13.14,Unknown_Unit,2025-06-20
P00052,CBC,3.0,mg/dL,2025-06-15
P00022,ECG,8.22,mg/dL,2025-08-08
P00149,COVID19 PCR,10.17,%,2025-07-22
P00101,COVID19 PCR,8.74,mg/dL,2025-06-22
P00054,HbA1c,11.98,%,2025-06-03


# Visits cleaning code

In [0]:
from pyspark.sql.functions import col, when

visits_df = spark.table("nilay_healthcare_catalog.bronze.visits")
display(visits_df)
print(visits_df.printSchema())

patient_id,admission_date,discharge_date,reason
P00129,2025-06-27,2025-06-28,Emergency
P00035,2025-05-25,2025-05-28,Checkup
P00052,2025-04-18,2025-04-22,Therapy
P00031,2025-08-05,2025-08-08,Checkup
P00104,2025-07-17,2025-07-26,Surgery
P00110,2025-05-10,2025-05-11,Surgery
P00021,2025-08-19,2025-08-29,Surgery
P00046,2025-07-04,2025-07-11,null
P00192,2025-06-20,2025-06-29,Therapy
P00157,2025-04-04,2025-04-09,Therapy


root
 |-- patient_id: string (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- reason: string (nullable = true)

None


# visits Anomalies
- `patient_id` contains null values and invalid IDs starting with **INVALID_**.
- `admission_date` contains null values and invalid IDs starting with **INVALID_**.
- `discharge_date` contains null values and invalid IDs starting with **INVALID_**.
- `reason` contains null values and invalid IDs starting with **INVALID_**.

In [0]:
# Step 1: Replace NULLs or INVALID_* values with 'Unknown'
visits_clean = visits_df.select([
    when((col(c).isNull()) | (col(c).rlike("^INVALID_.*")), "Unknown").otherwise(col(c)).alias(c)
    for c in visits_df.columns
])

# Step 2: Drop rows with Unknown in key fields
visits_clean = visits_clean.filter(
    (col("patient_id") != "Unknown") &
    (col("admission_date") != "Unknown") &
    (col("discharge_date") != "Unknown") &
    (col("reason") != "Unknown")
)

# Step 3: Drop duplicates
visits_clean = visits_clean.dropDuplicates()

# Step 4: Cast columns to proper types
visits_clean = visits_clean.select(
    col("patient_id").cast("string"),
    col("admission_date").cast("date"),
    col("discharge_date").cast("date"),
    col("reason").cast("string")
)

# Step 5: Save as cleaned Delta table
visits_clean.write \
  .mode("overwrite") \
  .format("delta") \
  .saveAsTable("nilay_healthcare_catalog.silver.visits_clean")

In [0]:
display(visits_clean)

patient_id,admission_date,discharge_date,reason
P00167,2025-07-01,2025-07-10,Surgery
P00199,2025-04-06,2025-04-16,Surgery
P00196,2025-06-29,2025-07-01,Checkup
P00060,2025-07-26,2025-07-30,Checkup
P00151,2025-06-17,2025-06-18,Surgery
P00120,2025-07-14,2025-07-17,Checkup
P00186,2025-06-15,2025-06-21,Emergency
P00140,2025-03-02,2025-03-07,Checkup
P00131,2025-05-14,2025-05-15,Emergency
P00147,2025-07-19,2025-07-24,Therapy


# Vitals Clean code

In [0]:
from pyspark.sql.functions import col, when, to_timestamp
vitals_df = spark.table("nilay_healthcare_catalog.bronze.vitals")

display(vitals_df)
print(vitals_df.printSchema())

patient_id,device_id,timestamp,vital_type,vital_value
P00175,DEV_KX9QY,2025-08-28T15:24:38.046556,glucose,92.1
P00061,DEV_WO79B,2025-08-28T14:25:30.046556,oxygen_saturation,95.3
INVALID_TT4,DEV_T2O63,2025-08-28T13:51:57.046556,heart_rate,75.2
P00092,DEV_24KOW,2025-08-29T03:08:33.046556,glucose,111.1
P00108,DEV_T2FGD,null,diastolic_bp,75.6
P00136,null,2025-08-28T18:42:27.046556,systolic_bp,120.3
P00071,DEV_TUFA1,2025-08-28T21:17:03.046556,glucose,78.9
P00030,DEV_9H5RO,2025-08-28T12:24:38.046556,heart_rate,73.7
P00186,DEV_W3JGK,2025-08-29T05:56:12.046556,systolic_bp,129.0
P00026,null,2025-08-28T18:38:33.046556,systolic_bp,130.0


root
 |-- patient_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- vital_type: string (nullable = true)
 |-- vital_value: double (nullable = true)

None


# Anomalies
- `patient_id` contains null values and invalid IDs starting with **INVALID_**.
- `device_id` contains null values and invalid IDs starting with **INVALID_**.
- `timestamp` contains null values and invalid IDs starting with **INVALID_**.
- `vital_type` contains null values and invalid IDs starting with **INVALID_**.
- `vital_value` contains null values and invalid IDs starting with **INVALID_**. It may also contains some outliers Cosidering vital_value of more than 500 as outlier.

In [0]:
vitals_df.sort(col("vital_value").desc()).display()

patient_id,device_id,timestamp,vital_type,vital_value
P00163,DEV_OTNEG,INVALID_T1W,diastolic_bp,5490.204799338022
P00065,DEV_AF2IX,2025-08-29T03:27:58.046556,null,5490.204799338022
P00134,DEV_6WTBF,2025-08-28T23:37:47.046556,heart_rate,5490.204799338022
P00026,DEV_BEKGE,2025-08-28T14:36:23.046556,oxygen_saturation,5490.204799338022
P00188,DEV_XF257,2025-08-29T06:57:01.046556,systolic_bp,5490.204799338022
P00025,DEV_EVP2M,2025-08-28T13:05:50.046556,diastolic_bp,5490.204799338022
P00018,DEV_B2UOF,2025-08-28T12:39:21.046556,systolic_bp,5490.204799338022
P00025,DEV_0OWKA,2025-08-28T11:22:26.046556,systolic_bp,5490.204799338022
P00006,null,2025-08-29T07:57:36.046556,systolic_bp,5490.204799338022
P00032,DEV_8VR73,2025-08-29T07:44:18.046556,glucose,5490.204799338022


In [0]:
# Step 1: Replace NULLs or INVALID_* with 'Unknown'
vitals_clean = vitals_df.select([
    when((col(c).isNull()) | (col(c).rlike("^INVALID_.*")), "Unknown").otherwise(col(c)).alias(c)
    for c in vitals_df.columns
])

# Step 2: Drop rows with Unknown in critical fields
vitals_clean = vitals_clean.filter(
    (col("patient_id") != "Unknown") &
    (col("device_id") != "Unknown") &
    (col("timestamp") != "Unknown") &
    (col("vital_type") != "Unknown") &
    (col("vital_value") != "Unknown")
)

# Step 3: Cast columns to correct data types
vitals_clean = vitals_clean.select(
    col("patient_id").cast("string"),
    col("device_id").cast("string"),
    col("timestamp").cast("timestamp"),
    col("vital_type").cast("string"),
    col("vital_value").cast("float")
)

# Step 4: Remove outliers in vital_value (valid range assumed: 0–500)
vitals_clean = vitals_clean.filter(
    (col("vital_value").isNotNull()) &
    (col("vital_value") >= 0) &
    (col("vital_value") <= 500)
)

# Step 5: Drop duplicates
vitals_clean = vitals_clean.dropDuplicates()

# Step 6: Save as cleaned Delta table
vitals_clean.write \
  .mode("overwrite") \
  .format("delta") \
  .saveAsTable("nilay_healthcare_catalog.silver.vitals_clean")

In [0]:
display(vitals_clean)

patient_id,device_id,timestamp,vital_type,vital_value
P00179,DEV_R4WMP,2025-08-29T09:09:06.046556Z,oxygen_saturation,99.4
P00117,DEV_UVW1E,2025-08-28T22:45:17.046556Z,oxygen_saturation,99.8
P00186,DEV_ZBW03,2025-08-28T23:48:39.046556Z,systolic_bp,151.6
P00050,DEV_6FAZR,2025-08-28T19:29:12.046556Z,glucose,78.5
P00063,DEV_L61KM,2025-08-28T15:17:30.046556Z,glucose,74.4
P00178,DEV_J18GF,2025-08-28T10:49:58.046556Z,heart_rate,125.6
P00130,DEV_H2XQ5,2025-08-28T21:49:04.046556Z,glucose,84.0
P00189,DEV_MFZE0,2025-08-29T02:21:38.046556Z,systolic_bp,125.6
P00092,DEV_ML469,2025-08-29T00:11:55.046556Z,heart_rate,116.1
P00193,DEV_KMSQN,2025-08-28T22:54:49.046556Z,systolic_bp,112.8


In [0]:
%sql 
show tables in silver

database,tableName,isTemporary
silver,labs_clean,false
silver,patient_doctor_map_clean,false
silver,patients_clean,false
silver,visits_clean,false
silver,vitals_clean,false
,_sqldf,true


# Gold layer

In [0]:
labs_clean.groupBy("test_name").count().show()

+------------+-----+
|   test_name|count|
+------------+-----+
|Unknown_Test|   49|
|         ECG|  172|
| Lipid Panel|  184|
|         CBC|  153|
| COVID19 PCR|  174|
|       HbA1c|  162|
+------------+-----+



There are mainly 5 types of test done on the lab

In [0]:
from pyspark.sql.functions import col, to_timestamp, avg, max, datediff

# Load cleaned tables
patients = spark.table("nilay_healthcare_catalog.silver.patients_clean")
visits = spark.table("nilay_healthcare_catalog.silver.visits_clean")
labs = spark.table("nilay_healthcare_catalog.silver.labs_clean")
vitals = spark.table("nilay_healthcare_catalog.silver.vitals_clean")
patient_doctor_map = spark.table("nilay_healthcare_catalog.silver.patient_doctor_map_clean")

# Add visit duration (discharge - admission)
visits = visits.withColumn(
    "visit_duration",
    datediff(col("discharge_date"), col("admission_date"))
)

# Lab summary → averages for multiple tests
lab_summary = (
    labs.filter(col("test_name").isin(["ECG", "Lipid Panel", "CBC", "COVID19 PCR", "HbA1c"]))
        .groupBy("patient_id")
        .pivot("test_name")
        .agg(avg("result"))
        .withColumnRenamed("ECG", "avg_ECG")
        .withColumnRenamed("Lipid Panel", "avg_LipidPanel")
        .withColumnRenamed("CBC", "avg_CBC")
        .withColumnRenamed("COVID19 PCR", "avg_COVID19PCR")
        .withColumnRenamed("HbA1c", "avg_HbA1c")
)

# Vitals summary → latest vital timestamp per patient
vital_summary = (
    vitals.groupBy("patient_id")
          .agg(max("timestamp").alias("latest_vital_timestamp"))
)

# Build Gold layer patient summary
gold_df = (
    patients
        .join(patient_doctor_map, "patient_id", "left")
        .join(visits, "patient_id", "left")
        .join(lab_summary, "patient_id", "left")
        .join(vital_summary, "patient_id", "left")
        .dropDuplicates()
)

# Write to Gold Delta table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nilay_healthcare_catalog.gold.gold_patient_summary")

In [0]:
%sql
select * from gold.gold_patient_summary

patient_id,name,age,gender,diagnosis,prescription,doctor_id,care_team,admission_date,discharge_date,reason,visit_duration,avg_CBC,avg_COVID19PCR,avg_ECG,avg_HbA1c,avg_LipidPanel,latest_vital_timestamp
P00155,Patient_TPUF,42,M,Heart Disease,DrugC,D003,Pulmonary,2025-05-16,2025-05-21,Therapy,5,null,7.745,null,null,null,2025-08-29T09:17:26.046556Z
P00199,Patient_JS2O,50,F,Asthma,DrugB,D001,Pulmonary,2025-04-06,2025-04-16,Surgery,10,null,5.665,null,6.65,null,2025-08-29T05:53:46.046556Z
P00189,Patient_W8NU,45,M,Diabetes,None,D004,General,2025-05-12,2025-05-19,Surgery,7,null,null,null,5.45,8.47,2025-08-29T09:01:36.046556Z
P00128,Patient_3WF1,56,Other,Undiagnosed,DrugC,D003,Pulmonary,2025-06-20,2025-06-29,Checkup,9,10.065,1.75,3.55,13.14,2.67,2025-08-29T09:02:49.046556Z
P00024,Patient_R0L9,27,F,None,DrugC,D001,General,2025-06-04,2025-06-10,Surgery,6,null,2.57,10.14,7.1,6.93,2025-08-29T09:08:03.046556Z
P00176,Patient_94Q4,50,F,Asthma,DrugA,D001,Cardio,2025-04-08,2025-04-12,Surgery,4,null,5.880000000000001,null,null,6.69,2025-08-29T07:19:11.046556Z
P00020,Patient_U4WK,44,Other,Diabetes,DrugA,D005,Endocrine,2025-06-08,2025-06-10,Emergency,2,null,null,10.07,8.1,4.24,2025-08-29T06:12:19.046556Z
P00092,Patient_3AL6,28,Other,Asthma,None,null,null,2025-03-07,2025-03-15,Checkup,8,8.38,null,null,null,10.92,2025-08-29T09:10:23.046556Z
P00107,Patient_6BDG,45,Other,Heart Disease,None,D003,Cardio,2025-05-07,2025-05-08,Surgery,1,null,0.87,null,5.68,null,2025-08-29T09:19:39.046556Z
P00196,Patient_K0TI,49,F,Asthma,DrugB,D002,General,2025-06-29,2025-07-01,Checkup,2,7.73,null,null,201.5862202688728,3.91,2025-08-29T05:37:52.046556Z
